In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import spacy

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import json

In [3]:
import sys

print(sys.version)
print(sys.executable)
print(spacy.__version__)

3.12.12 | packaged by conda-forge | (main, Oct 22 2025, 23:34:53) [Clang 19.1.7 ]
/Users/Akseldkw/micromamba/envs/kret_312/bin/python
3.8.11


In [4]:
# %pip install pyarrow
# %pip install -U pyarrow pandas

In [5]:
# pip install pandas

# if you ran intp 3.9 python version problem in vscode, do:
# /opt/homebrew/bin/python3.10 -m pip install --upgrade pip
# /opt/homebrew/bin/python3.10 -m pip install ipykernel
# /opt/homebrew/bin/python3.10 -m ipykernel install --user --name=python310 --display-name "Python 3.10"

# then in top right, switch Python 3.10.*

In [6]:
# %pip install --upgrade pip
# %pip install torch torchvision torchaudio sentence-transformers
# %pip install spacy==3.7.1
# %pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl

In [32]:
from uml_project import *

# note: constants.py requires python version >= 3.10
HF_DIR, HF_REGISTRY, MODEL_DIR, DATA_DIR, ROOT_DIR

(PosixPath('/Users/Akseldkw/coding/Columbia/UML-Project/data/huggingface'),
 PosixPath('/Users/Akseldkw/coding/Columbia/UML-Project/data/huggingface/REGISTRY.json'),
 PosixPath('/Users/Akseldkw/coding/Columbia/UML-Project/models'),
 PosixPath('/Users/Akseldkw/coding/Columbia/UML-Project/data'),
 PosixPath('/Users/Akseldkw/coding/Columbia/UML-Project'))

In [33]:
nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("sentencizer")

In [34]:
tsne_tex_path = Path(DATA_DIR / "scientific/stochastic_neighbor_embedding_under_f_divergences__discussion.json")

In [35]:
tsne_tex_dict = json.loads(tsne_tex_path.read_text())

In [36]:
tsne_tex_dict.keys()

dict_keys(['authors', 'date_published', 'raw_tex', 'title'])

In [37]:
tsne_sents1 = define_sentence(tsne_tex_dict["raw_tex"], nlp=nlp, latex_mode="sentences")

In [38]:
tsne_sents1

array(['Other divergences for -SNE optimization have been explored previously Perhaps the first detailed study was done by (Gamma- Bregman- and -divergences) and their corresponding visualizations on some image processing datasets different divergences can be used to find micro and macro relationships in data',
       ', but do not focus on faithful discovery of intrinsic structures An interesting line of work by PAPER and discovery and multi-scale visualizations to find local and global structures The work by PAPER is closely related where they study -divergences from an informational retrieval perspective',
       'Our work extends it to the general class of -divergences and explores the relationships between data structure and the type of divergence used It is worth emphasizing that no previous study makes an explicit connection between the choice of divergence and the type of structure discovery',
       'Our work makes this explicit and should help a practitioner gain better insig

In [39]:
tsne_sents2 = define_sentence(tsne_tex_dict["raw_tex"], nlp=nlp, latex_mode="strip")

In [40]:
tsne_sents2

array(['Other divergences for -SNE optimization have been explored previously Perhaps the first detailed study was done by (Gamma- Bregman- and -divergences) and their corresponding visualizations on some image processing datasets different divergences can be used to find micro and macro relationships in data',
       'Our work extends it to the general class of -divergences and explores the relationships between data structure and the type of divergence used It is worth emphasizing that no previous study makes an explicit connection between the choice of divergence and the type of structure discovery',
       'Our work makes this explicit and should help a practitioner gain better insights about their data in the data exploration phase Our work goes a step further and attempts to ameliorate the issues non-convex objective function in the -SNE criterion By studying the variational dual form, we can achieve better quality (locally optimal) solutions, which would be extremely beneficial 

# Verma Papers Preprocessing

In [41]:
with open(ROOT_DIR / "uml_project/data/pre_processing/tmp-verma.txt", "r", encoding="utf-8") as f:
    list_of_paths = f.read().splitlines()

In [42]:
print(list_of_paths[:10])

list_of_paths = list_of_paths[
    1:
]  # ignore /Users/alenachan/Desktop/Columbia/Fall-2025/UML/UML/data/scientific/REGISTRY.json

['/Users/alenachan/Desktop/Columbia/Fall-2025/UML/UML/data/scientific/REGISTRY.json', '/Users/alenachan/Desktop/Columbia/Fall-2025/UML/UML/data/scientific/a_neural_network_solves_explains_and_generates_university_math_problems_by_program_synthesis_and_few_shot_learning_at_human_level__generation_18_01.json', '/Users/alenachan/Desktop/Columbia/Fall-2025/UML/UML/data/scientific/a_neural_network_solves_explains_and_generates_university_math_problems_by_program_synthesis_and_few_shot_learning_at_human_level__generation_18_02.json', '/Users/alenachan/Desktop/Columbia/Fall-2025/UML/UML/data/scientific/a_neural_network_solves_explains_and_generates_university_math_problems_by_program_synthesis_and_few_shot_learning_at_human_level__generation_18_03.json', '/Users/alenachan/Desktop/Columbia/Fall-2025/UML/UML/data/scientific/a_neural_network_solves_explains_and_generates_university_math_problems_by_program_synthesis_and_few_shot_learning_at_human_level__generation_18_05.json', '/Users/alenachan/

In [43]:
OUTPUT_DIR = DATA_DIR / "scientific/processed"
OUTPUT_PARQUET = OUTPUT_DIR / "papers.parquet"
METADATA_FILE = OUTPUT_DIR / "metadata.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for pth in list_of_paths:
    # print(pth)
    pth = pth.replace("/Users/alenachan/Desktop/Columbia/Fall-2025/UML/UML/data", DATA_DIR.as_posix())
    doc_id = os.path.splitext(os.path.basename(pth))[0]

    tsne_tex_path = Path(pth)
    tsne_tex_dict = json.loads(tsne_tex_path.read_text())
    tsne_sents = define_sentence(tsne_tex_dict["raw_tex"], nlp=nlp, latex_mode="sentences")

    if len(tsne_sents) == 0:
        print(f"Skipping {doc_id}: tsne_sents returned empty")
        continue

    records = []

    for i, sent in enumerate(tsne_sents):
        records.append(
            {
                "corpus": "scientific_papers",
                "doc_id": doc_id,
                "sent_index": i,
                "sentence": sent,
                "split": "train",  # could later vary by dataset
                "token_count": len(sent.split()),
            }
        )

    df = pd.DataFrame(records)
    table = pa.Table.from_pandas(df)
    pq.write_table(table, OUTPUT_PARQUET)

    metadata = {
        "corpus_name": "scientific_papers",
        "description": "Sentence-level dataset built from parsed scientific papers.",
        "num_documents": len(df["doc_id"].unique()),
        "num_sentences": len(df),
        "columns": df.columns.tolist(),
        "example_entry": df.iloc[0].to_dict() if len(df) else {},
    }

    with open(METADATA_FILE, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4)

Skipping an_analysis_of_document_graph_construction_methods_for_amr_summarization__alg_person_merge: tsne_sents returned empty
Skipping an_analysis_of_document_graph_construction_methods_for_amr_summarization__fig_example: tsne_sents returned empty
Skipping an_analysis_of_document_graph_construction_methods_for_amr_summarization__fig_pipeline: tsne_sents returned empty
Skipping an_analysis_of_document_graph_construction_methods_for_amr_summarization__tab_merge: tsne_sents returned empty
Skipping main__appendixc: tsne_sents returned empty
Skipping main__appendixd: tsne_sents returned empty
Skipping main__appendixe: tsne_sents returned empty
Skipping main__appendixg: tsne_sents returned empty
Skipping main__appendixo: tsne_sents returned empty
Skipping main__appendixp: tsne_sents returned empty
Skipping main__appendixq: tsne_sents returned empty
Skipping main__appendixr: tsne_sents returned empty
Skipping main__appendixs: tsne_sents returned empty
Skipping main__appendixt: tsne_sents ret

In [44]:
df = pd.read_parquet(DATA_DIR / "scientific/processed/papers.parquet")
print(df.head())

              corpus                                             doc_id  \
0  scientific_papers  t_sne_exaggerates_clusters_provably__related_work   
1  scientific_papers  t_sne_exaggerates_clusters_provably__related_work   
2  scientific_papers  t_sne_exaggerates_clusters_provably__related_work   
3  scientific_papers  t_sne_exaggerates_clusters_provably__related_work   
4  scientific_papers  t_sne_exaggerates_clusters_provably__related_work   

   sent_index                                           sentence  split  \
0           0  There are two notably distinct ways of context...  train   
1           1  In a chronological sense Confidence in the dat...  train   
2           2  Some works argue that these methods have merit...  train   
3           3  PAPER , for instance, characterized the distin...  train   
4           4  PAPER proved a consistency result for a contin...  train   

   token_count  
0           16  
1           24  
2           58  
3           26  
4           2

# Taylor Preprocessing

In [45]:
nlp = build_sentencizer(True)  # or True if you've installed en_core_web_sm
text = Path(DATA_DIR / "music/taylor_swift/Albums/Red/IKnewYouWereTrouble.txt").read_text()
sents = define_sentence(text, nlp=nlp)

In [46]:
from pathlib import Path

OUTPUT_DIR = DATA_DIR / "music/processed"
OUTPUT_PARQUET = OUTPUT_DIR / "songs.parquet"
METADATA_FILE = OUTPUT_DIR / "metadata.json"

albums_root = Path(DATA_DIR / "music/taylor_swift/Albums")
txt_paths = sorted(albums_root.rglob("*.txt"))
if not txt_paths:
    raise FileNotFoundError(f"No .txt files found under {albums_root}")

records = []

for txt_path in txt_paths:
    album_name = txt_path.parent.name
    song_id = txt_path.stem
    text = txt_path.read_text(encoding="utf-8")
    song_sents = define_sentence(text, nlp=nlp)
    if len(song_sents) == 0:
        print(f"Skipping {txt_path}: no valid sentences")
        continue
    for sent_index, sent in enumerate(song_sents):
        records.append(
            {
                "corpus": "taylor_swift_albums",
                "album": album_name,
                "doc_id": song_id,
                "sent_index": sent_index,
                "sentence": sent,
                "split": "train",
                "token_count": len(sent.split()),
            }
        )

if not records:
    raise ValueError("Taylor Swift album corpus is empty after filtering.")

df = pd.DataFrame(records)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
table = pa.Table.from_pandas(df)
pq.write_table(table, str(OUTPUT_PARQUET))

metadata = {
    "corpus_name": "taylor_swift_albums",
    "description": "Sentence-level dataset built from Taylor Swift album lyrics (.txt files).",
    "num_documents": int(df["doc_id"].nunique()),
    "num_sentences": int(len(df)),
    "columns": df.columns.tolist(),
    "example_entry": df.iloc[0].to_dict(),
}

METADATA_FILE.write_text(json.dumps(metadata, indent=4), encoding="utf-8")
print(f"Wrote {metadata['num_sentences']} sentences from {metadata['num_documents']} songs to {OUTPUT_PARQUET}")

Skipping /Users/Akseldkw/coding/Columbia/UML-Project/data/music/taylor_swift/Albums/Reputation/ReputationMagazineVol_1.txt: no valid sentences
Wrote 47264 sentences from 392 songs to /Users/Akseldkw/coding/Columbia/UML-Project/data/music/processed/songs.parquet


In [47]:
music_df = pd.read_parquet(DATA_DIR / "music/processed/songs.parquet")
music_df.sample()

,corpus,album,doc_id,sent_index,sentence,split,token_count
1910,taylor_swift_albums,1989_TaylorsVersion__Deluxe_,AllYouHadToDoWasStay_TaylorsVersion_,14,But I don't know what to say,train,7


In [48]:
f_embed = music_df.sentence.str.contains("Embed")

In [49]:
music_df.sentence[0]

'The drought was the very worst (Oh-oh, oh-oh)'

In [50]:
f_embed.sum()

np.int64(0)

In [51]:
music_df[f_embed].sentence.tolist()

[]